In [1]:
'''
This assignment was completed with assistance of Claude Code.
'''

'\nThis assignment was completed with assistance of Claude Code.\n'


---

## **Part 1 — Exploring Different Modalities / Representations of Network Traffic**

Synthetic network traffic generation is useful for many applications such as dataset augmentation, network testing, and resource management. Many existing generation methods treat traffic generation as either a time-series prediction task or an autoregressive modeling task. In these approaches, models are trained directly on structured representations of packets—one common example is **nPrint**, a tabular format that encodes packet header fields from PCAP traces into machine-learning-friendly numerical vectors.

In an **nPrint**, each row represents one packet in a trace. The entire nPrint file (a large CSV) represents all packets of that trace in sequential order. Header fields are encoded using one-hot–like binary indicators (e.g., `1`, `0`, or `01`), making them easy to feed into ML models.

However, general-purpose ML models—even advanced time-series architectures and transformers—often struggle to capture the *complex dependencies* present in real network traffic:

* **Local dependencies**: relationships among columns within a single row (i.e., dependencies among header fields of a single packet).
* **Global dependencies**: relationships across rows (i.e., the evolution of packets within the same flow).
  For example: if the first packet of a flow uses TCP, the second packet in that same flow should also be TCP; sequence numbers, flags, and flow identifiers evolve in structured ways over time.

While sequential ML models struggle with these multi-scale dependencies, **vision models** (e.g., diffusion models) excel at capturing both local and global structure when data is presented spatially—like an image. By converting nPrint traces into 2D PNG images, we can take advantage of the strong representational capabilities of image models and generate synthetic traffic using visual generative approaches.

---

### **Your Task for Part 1**

In this part of the assignment, you will:

1. **Inspect the raw nPrint files** (found in the `real_nprints` directory).
2. **Understand how each CSV row and column corresponds to packet-level metadata.**
3. **Follow the provided conversion pipeline** that transforms these nPrint CSVs into PNG image representations suitable for use with diffusion models and other visual architectures.
4. **Take an already generated set of image-representation of images and convert them back into nprint representation for downstream task utilization**

This will help you understand why converting network traces to images can unlock generative modeling capabilities that traditional ML approaches struggle with.

Q1:
First, download and unzip the data you will need from (https://drive.google.com/file/d/1hY6nNXEYOwl1l-O_nCknO9xezcHr6ZXi/view?usp=sharing)
In your own words, describe how an nPrint CSV encodes a network trace.
Why might a 2D image representation capture structural relationships that a row-by-row CSV cannot?

nPrint works by taking a packet and turning it into a bit table. For each of the bits in the packet's headers, there's a dedicated column for it in the table. If the bit is set, there is a 1. If it is not set, it is a 0. If it doesn't exist, it is a -1. Thus, nPrint provides a standardized way to organize packet information. 

A 2D image representation is helpful because it can help show things that change over time more easily. Compared to a CSV file, in which a machine might have to look at the data row by row, the 2D image allows the comoputer to better understand what changes over time. This is helpful in recognizing patterns that could only be found in seeing the entire "conversation" over time. 

Q2. Design a method for converting nPrint representations of traces (in the folder real_nprints) into image representations.
Your image representation should use only the first 1024 packets from each trace to avoid producing images that are too large.
Save the images into a folder called './nd_data/student_converted_images'

I will make 1 red, 0 black, and -1 white. In this, we can see the images really easily. 

In [2]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

In [3]:
colors = {1: (255, 0, 0), 0: (0, 0, 0), -1: (255, 255, 255)}

def nprint_to_image(nprint_path):
    df = pd.read_csv(nprint_path, index_col=0)
    df = df.iloc[:1024, :]

    height, width = df.shape
    arr = df.to_numpy()

    img = np.zeros((height, width, 3), dtype=np.uint8)
    for value, rgb in colors.items():
        img[arr == value] = rgb

    return Image.fromarray(img, 'RGB')

In [4]:
nprint_files = glob.glob(os.path.join("real_nprints", "*.nprint"))

for path in nprint_files:
    name = os.path.splitext(os.path.basename(path))[0]
    img = nprint_to_image(path)
    img.save(os.path.join("nd_data/student_converted_images", f"{name}.png"))

In [ ]:
'''
A lot of the file paths were wrong and broke my code, so I put them all into the nd_data folder instead.
'''

In [5]:
# The following is a pre-defined script used in NetDiffusion that will convert all of the provided real nprints into image representations. Run this code and observe the output
!python ./scripts/nprint_to_png.py -i ./nd_data/real_nprints/ -o ./nd_data/real_traffic_images

Processing amazon_1024_7.nprint
/home/hoath/ml_for_systems/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing zoom_1024_13.nprint
/home/hoath/ml_for_systems/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing teams_1024_17.nprint
/home/hoath/ml_for_systems/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing instagram_1024_15.nprint
/home/hoath/ml_for_systems/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processin

Q3: Now that you have seen how NetDiffusion converts nPrints into images, compare their method with the approach you designed in Q2. What are the advantages and disadvantages of each, especially in terms of what might help or hinder a vision model’s ability to learn?

NetDiffusion uses RGB instead of the red, black, and white color scheme that I used. I think that this makes more sense from a computational perspective because these colors are the furthest apart, so it's easier for a computer to recognize them as distinct classes. Meanwhile, for mine, black and white can be made using red, so there could possibly be feature contamination there. For humans, though, I think that the red, black, and white makes the images more clear in terms of what featuresa are there and what is not. NetDiffusion also has fallback in case nprint doesn't work correctly, which is something I didn't consider.


---

## **Part 2 — Converting Generated Images Back Into Usable Format**

In the first part of the assignment, you explored how network traces in nPrint format can be transformed into image representations suitable for vision-based generative models. In Part 2, we focus on the reverse process: taking synthetic images produced by these models and converting them back into structured network representations.

This step is crucial because real-world applications do not operate on images—they require valid, interpretable packet traces that can be analyzed, replayed, or integrated into downstream tools.

---

### **Your Task for Part 2**

In this part of the assignment, you will:

1. **Convert generated images back into the original nPrint representation.**
   You will follow a scripted pipeline that translates pixel intensities and color channels back into binary header fields, reconstructing the packet-level structure of the trace.

2. **Apply essential post-processing techniques to correct errors introduced by diffusion models.**
   Generated images are rarely perfect—vision models may introduce color drift, pixel misalignment, noise, or structural artifacts.
   You will observe how heuristic correction, formatting enforcement, and reconstruction steps ensure that the converted nPrints become:

   * syntactically valid,
   * structurally consistent,
   * and replayable.

Across this section, your goal is to understand **why the reverse transformation is fragile**, which types of artifacts break reversibility, and how post-processing logic helps repair or compensate for generative errors.

For simplicity of this assignment, we have trainined and generated the images for you. If you have sufficient GPU access and want to try fine-tuning the model and generating the images yourself, feel free to take a look at the public repo (https://github.com/noise-lab/NetDiffusion).


Q4: We have taken the images converted by NetDiffusion and trained a LoRA-fine-tuned Stable Diffusion model (with ControlNet) to generate synthetic traffic images for you. These generated samples are stored in generated_traffic_images/.
Compare these generated images visually with the real images you saw earlier in real_traffic_images/.

Do you notice anything different between the real and generated traffic images? What immediately stands out as potentially problematic if we attempt to convert these generated images back into nPrint format? (Descriptive Only)

The first is that the pixels aren't really clear and the color scheme is different. For example, the "0"s are purple in the generated image versus blue in the NetDiffusion images, so if we use a straightforward blue to 0 converter, it would fail. In addition, there is a patch of green to the right of the image which wasn't there in the oridginal image. 

Q5: If you were to design a method to convert generated images back into nPrints, how would you do it? Explain your approach and describe how your method addresses the concerns you raised in Q4. (Descriptive only)

I would go through each pixel and ensure that they are formatted correctly first (the color is what I am expecting) and convert those that are not by using some kind of threshold via RGB scale. I would make sure there's no blend of colors and figure out a way to make sure that blends are taken into account.

Below are a set of pre-written scripts that perform the necessary post-generation augmentation and processing on the synthetic images you obtained from the diffusion model. These scripts handle tasks such as color normalization/augmentation and conversion from generated images back into nPrint format.
(The PCAP step is optional — you may run it if you are interested in observing or replaying the reconstructed traffic.)

In [6]:
# Step 1: Color Augmentation
!python ./scripts/color_processor.py \
  --input_dir="./nd_data/generated_traffic_images" \
  --output_dir="./nd_data/color_corrected_generated_traffic_images"

Processed 100 images.


In [7]:
# Step 2: Image-to-nPrint Conversion
!python ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ./nd_data/color_corrected_generated_traffic_images \
  --output_dir ./nd_data/generated_nprint

Processing ./nd_data/color_corrected_generated_traffic_images/teams_1.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_1.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/zoom_1.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_1.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/teams_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_6.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/netflix_5.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/netflix_5.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitter_8.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitter_8.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitch_0.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_0.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitch_2.png with size 1088 x 1024
Saved ./nd_data/generated_nprin

Q6: You may now read through the provided scripts in color_processor.py (color augmentation / normalization) and image_to_nprint.py (image → nPrint reconstruction).
How do the post-processing methods implemented in these scripts compare to the approach you proposed in Q5?
Describe the pros and cons of both methods and highlight any differences in design philosophy, robustness, or assumptions.

The post-processing method is very similar, but is much more comprehensive than the one that I initially thuought of. The biggest change is the inclusion of code that deals with what happens if there are pixels that do not fall into the thresholds that I thought of. The code solves this by taking the max color, which allows the pixel to be categorized into the most likely color. I think the biggest con is that it's unclear how the machine decided these thresholds and if there is a better threshold to use instead. Overall, the method provided is much more robust than the one I provided. 


---

# **Part 3 — Using Real and Synthetic nPrints for Application Classification**

In the previous parts, you learned how network traces can be converted between nPrint and image representations, generated using diffusion models, and reconstructed back into nPrint format.
Now, you will evaluate how useful these generated nPrints are for downstream **machine learning tasks**.

Each nPrint file—whether real or generated—is labeled with the **application** that produced the traffic (e.g., `amazon_1.nprint` means this sample came from Amazon traffic).
In this section, you will treat each **entire nPrint file as a single sample** and build a simple ML pipeline to classify application labels.

To simplify the task, you will restrict your model to use **only the first 3 packets** (3 rows) from each nPrint.
This mimics “early packet classification,” where only the beginning of a flow is available.

---


Q7:

You now have access to both `real_nprints/` and `generated_nprint/`.
Notice that in both directories, files are labeled using the application associated with that nPrint (e.g., `amazon_1.nprint`).
Treat each **nPrint file** as one sample.

**Design an ML pipeline that trains a model using *synthetic nPrints* (from `generated_nprint/`) and evaluates its performance on *real nPrints* (from `real_nprints/`) to predict the correct application label.**

Your pipeline should:

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.

We have already written the script to load the real and generated nprints into DataFrame for you.

In [8]:
import os
import glob
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Safe conversion for nPrint cell
# ---------------------------------------------------------------
def safe_convert(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()

    if x in ["0", "1", "-1"]:
        return int(x)

    if all(c in "01" for c in x) and len(x) <= 16:
        return int(x, 2)

    if x.lstrip("-").isdigit():
        return int(x)

    return 0


# ---------------------------------------------------------------
# Get original column names from a reference nPrint
# ---------------------------------------------------------------
def get_original_columns(example_path="nd_data/real_nprints"):
    first_file = glob.glob(os.path.join(example_path, "*.nprint"))[0]

    df = pd.read_csv(first_file)

    # Drop index column if present (like "Unnamed: 0")
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    return list(df.columns)


# ---------------------------------------------------------------
# Load nPrint → first 3 rows → flatten with prefixed column names
# ---------------------------------------------------------------
def load_nprint_with_colnames(path, base_cols, num_rows=3):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Drop "Unnamed: 0" if present
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    df = df.iloc[:num_rows, :]            # first 3 packets  
    df = df.map(safe_convert)             # clean convert  

    # Build prefixed column names
    pkt_cols = []
    for pkt in range(1, num_rows + 1):
        pkt_cols.extend([f"pkt{pkt}_{c}" for c in base_cols])

    # Flatten 3×columns into 1 vector
    flat = df.values.flatten()

    return flat, pkt_cols


# ---------------------------------------------------------------
# Load entire directory into a DataFrame (with labels)
# ---------------------------------------------------------------
def load_directory_as_df(directory, base_cols):
    rows = []
    labels = []
    colnames_set = None

    for path in glob.glob(os.path.join(directory, "*.nprint")):
        label = os.path.basename(path).split("_")[0]

        flat, cn = load_nprint_with_colnames(path, base_cols)
        rows.append(flat)
        labels.append(label)

        if colnames_set is None:
            colnames_set = cn   # only set once

    df = pd.DataFrame(rows, columns=colnames_set)
    df["label"] = labels
    return df


# ---------------------------------------------------------------
# FINAL: Load real + synthetic DataFrames
# ---------------------------------------------------------------
base_cols = get_original_columns("nd_data/real_nprints")

df_synth = load_directory_as_df("nd_data/generated_nprint", base_cols)
df_real  = load_directory_as_df("nd_data/real_nprints", base_cols)

print("Synthetic DF:", df_synth.shape)
print(df_synth.head())

print("\nReal DF:", df_real.shape)
print(df_real.head())


Synthetic DF: (100, 3265)
   pkt1_ipv4_ver_0  pkt1_ipv4_ver_1  pkt1_ipv4_ver_2  pkt1_ipv4_ver_3  \
0                1                0                0                0   
1                0                0                0                0   
2                1                1                1                1   
3                0                0                0                0   
4                1                0                0                0   

   pkt1_ipv4_hl_0  pkt1_ipv4_hl_1  pkt1_ipv4_hl_2  pkt1_ipv4_hl_3  \
0               0               0               0               1   
1               0               0               1               1   
2               1               1               1               0   
3               0               0               0               1   
4               0               0               1               1   

   pkt1_ipv4_tos_0  pkt1_ipv4_tos_1  ...  pkt3_icmp_roh_23  pkt3_icmp_roh_24  \
0                0                0  ...

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [10]:
X = df_synth.drop(columns=["label"])
y = df_synth["label"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [13]:
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      amazon       1.00      1.00      1.00         1
    facebook       1.00      1.00      1.00         2
        meet       1.00      1.00      1.00         3
       teams       0.00      0.00      0.00         1
      twitch       1.00      1.00      1.00         3
     twitter       1.00      1.00      1.00         3
     youtube       1.00      1.00      1.00         5
        zoom       0.67      1.00      0.80         2

    accuracy                           0.95        20
   macro avg       0.83      0.88      0.85        20
weighted avg       0.92      0.95      0.93        20



/home/hoath/ml_for_systems/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hoath/ml_for_systems/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hoath/ml_for_systems/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [14]:
print(confusion_matrix(y_test, y_pred))

[[1 0 0 0 0 0 0 0]
 [0 2 0 0 0 0 0 0]
 [0 0 3 0 0 0 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 3 0 0 0]
 [0 0 0 0 0 3 0 0]
 [0 0 0 0 0 0 5 0]
 [0 0 0 0 0 0 0 2]]


In [15]:
X_real = df_real.drop(columns=["label"])
y_real = df_real["label"]

In [16]:
y_pred_real = rf.predict(X_real)
print(classification_report(y_real, y_pred_real))

              precision    recall  f1-score   support

      amazon       0.00      0.00      0.00        20
    facebook       0.40      0.10      0.16        20
   instagram       0.12      0.05      0.07        20
        meet       0.27      0.30      0.29        20
     netflix       0.21      0.65      0.32        20
       teams       0.20      0.65      0.31        20
      twitch       0.25      0.10      0.14        20
     twitter       0.40      0.50      0.44        20
     youtube       0.33      0.05      0.09        20
        zoom       0.00      0.00      0.00        20

    accuracy                           0.24       200
   macro avg       0.22      0.24      0.18       200
weighted avg       0.22      0.24      0.18       200



/home/hoath/ml_for_systems/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hoath/ml_for_systems/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/hoath/ml_for_systems/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

In [17]:
'''
Accuracy drops from .95 to .24 when moving from predicting on syn data test set to predicting on real data. This means that there is something useful about
the synth data, or at least it's better than guessing. Given that .95 accuracy was a pretty strong signal, I think that the problem lies in the NetDiffusion
generation and that there probably needs to be a better model to produce synths. Something interesting in the initial training on synth data is that 
precision for teams was 0, which might mean that there is not enough comprehensive data in the oridginal dataset to begin with, and that might have affected
accuracy in the later predictions on real nprints.
'''

"\nAccuracy drops from .95 to .24 when moving from predicting on syn data test set to predicting on real data. This means that there is something useful about\nthe synth data, or at least it's better than guessing. Given that .95 accuracy was a pretty strong signal, I think that the problem lies in the NetDiffusion\ngeneration and that there probably needs to be a better model to produce synths. Something interesting in the initial training on synth data is that \nprecision for teams was 0, which might mean that there is not enough comprehensive data in the oridginal dataset to begin with, and that might have affected\naccuracy in the later predictions on real nprints.\n"